# COMI Microscopy Image Classification

This notebook trains and evaluates two models on the COMI `BPAEC` dataset:

- a baseline custom CNN
- a transfer-learning `ResNet18`

Manual steps you still need to do in Colab:

1. Mount Google Drive
2. Update `PROJECT_ROOT` and `DATASET_ROOT`
3. Turn on the GPU runtime
4. Run the cells in order

No manual labeling is required.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Google Colab drive mount skipped. Update local paths manually if needed.')

PROJECT_ROOT = Path('/content/drive/MyDrive/comi_project')
DATASET_ROOT = Path('/content/drive/MyDrive/ME-494 Project Test 2/COMI/dataset/BPAEC')
RESULTS_ROOT = PROJECT_ROOT / 'results'
FIGURES_ROOT = RESULTS_ROOT / 'figures'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('RESULTS_ROOT =', RESULTS_ROOT)

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt'),
])

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import pandas as pd
from IPython.display import display

from comi_pipeline import (
    CLASS_NAMES,
    TrainingConfig,
    build_model,
    build_split_manifest,
    compare_model_metrics,
    create_dataloaders,
    ensure_dir,
    evaluate_and_save,
    fit_model,
    plot_class_distribution,
    plot_confusion_matrix,
    plot_gradcam_examples,
    plot_prediction_examples,
    plot_sample_images,
    plot_training_history,
    resolve_device,
    save_manifest,
    seed_everything,
    summarize_manifest,
)

seed_everything(42)
device = resolve_device()
ensure_dir(RESULTS_ROOT)
ensure_dir(FIGURES_ROOT)
print('Using device:', device)

In [ ]:
manifest = build_split_manifest(DATASET_ROOT)
summary = summarize_manifest(manifest)

save_manifest(manifest, RESULTS_ROOT / 'comi_split_manifest.csv')
summary.to_csv(RESULTS_ROOT / 'comi_split_summary.csv')

display(summary)

plot_class_distribution(manifest, FIGURES_ROOT / 'class_distribution.png')
plot_sample_images(manifest, samples_per_class=3, split='train', output_path=FIGURES_ROOT / 'sample_train_images.png')

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_loaders, _, smoke_manifest = create_dataloaders(
        manifest,
        image_size=224,
        batch_size=16,
        num_workers=2,
        normalize_mode='simple',
        smoke_test=True,
    )
    smoke_config = TrainingConfig(
        model_name='baseline_cnn',
        batch_size=16,
        num_epochs=2,
        early_stopping_patience=2,
        normalize_mode='simple',
    )
    smoke_model = build_model('baseline_cnn', num_classes=len(CLASS_NAMES), pretrained=False)
    smoke_model, smoke_history = fit_model(
        smoke_model,
        smoke_loaders,
        smoke_config,
        output_dir=RESULTS_ROOT / 'smoke_test',
        device=device,
        pretrained=False,
    )
    display(smoke_manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0))
    display(smoke_history)
else:
    print('Smoke test skipped. Set RUN_SMOKE_TEST = True to run a quick end-to-end check.')

In [ ]:
baseline_loaders, baseline_datasets, _ = create_dataloaders(
    manifest,
    image_size=224,
    batch_size=32,
    num_workers=2,
    normalize_mode='simple',
)

baseline_config = TrainingConfig(
    model_name='baseline_cnn',
    image_size=224,
    batch_size=32,
    num_epochs=20,
    learning_rate=1e-3,
    early_stopping_patience=5,
    normalize_mode='simple',
)

baseline_model = build_model('baseline_cnn', num_classes=len(CLASS_NAMES), pretrained=False)
baseline_model, baseline_history = fit_model(
    baseline_model,
    baseline_loaders,
    baseline_config,
    output_dir=RESULTS_ROOT / 'baseline_cnn',
    device=device,
    pretrained=False,
)

display(baseline_history.tail())
plot_training_history(baseline_history, 'Baseline CNN', FIGURES_ROOT / 'baseline_history.png')

baseline_predictions, baseline_metrics, baseline_report = evaluate_and_save(
    baseline_model,
    baseline_loaders['test'],
    output_dir=RESULTS_ROOT / 'baseline_cnn',
    prefix='baseline_test',
    device=device,
)
display(pd.DataFrame([baseline_metrics]).drop(columns=['confusion_matrix', 'classification_report']))
display(baseline_report)
plot_confusion_matrix(baseline_metrics, CLASS_NAMES, FIGURES_ROOT / 'baseline_confusion_matrix.png')
plot_prediction_examples(baseline_predictions, samples=6, correct=False, output_path=FIGURES_ROOT / 'baseline_misclassified.png')
plot_prediction_examples(baseline_predictions, samples=6, correct=True, output_path=FIGURES_ROOT / 'baseline_correct.png')

In [ ]:
resnet_loaders, resnet_datasets, _ = create_dataloaders(
    manifest,
    image_size=224,
    batch_size=32,
    num_workers=2,
    normalize_mode='imagenet',
)

resnet_config = TrainingConfig(
    model_name='resnet18',
    image_size=224,
    batch_size=32,
    num_epochs=20,
    learning_rate=1e-3,
    fine_tune_learning_rate=1e-4,
    warmup_epochs=2,
    early_stopping_patience=5,
    normalize_mode='imagenet',
)

resnet_model = build_model('resnet18', num_classes=len(CLASS_NAMES), pretrained=True)
resnet_model, resnet_history = fit_model(
    resnet_model,
    resnet_loaders,
    resnet_config,
    output_dir=RESULTS_ROOT / 'resnet18',
    device=device,
    pretrained=True,
)

display(resnet_history.tail())
plot_training_history(resnet_history, 'ResNet18', FIGURES_ROOT / 'resnet18_history.png')

resnet_predictions, resnet_metrics, resnet_report = evaluate_and_save(
    resnet_model,
    resnet_loaders['test'],
    output_dir=RESULTS_ROOT / 'resnet18',
    prefix='resnet18_test',
    device=device,
)
display(pd.DataFrame([resnet_metrics]).drop(columns=['confusion_matrix', 'classification_report']))
display(resnet_report)
plot_confusion_matrix(resnet_metrics, CLASS_NAMES, FIGURES_ROOT / 'resnet18_confusion_matrix.png')
plot_prediction_examples(resnet_predictions, samples=6, correct=False, output_path=FIGURES_ROOT / 'resnet18_misclassified.png')
plot_prediction_examples(resnet_predictions, samples=6, correct=True, output_path=FIGURES_ROOT / 'resnet18_correct.png')

In [ ]:
comparison = compare_model_metrics({
    'baseline_cnn': baseline_metrics,
    'resnet18': resnet_metrics,
})
comparison.to_csv(RESULTS_ROOT / 'model_comparison.csv', index=False)
display(comparison)

gradcam_candidates = resnet_predictions.loc[resnet_predictions['is_correct']].head(6).copy()
if gradcam_candidates.empty:
    gradcam_candidates = resnet_predictions.head(6).copy()

plot_gradcam_examples(
    resnet_model,
    resnet_datasets['test'],
    gradcam_candidates,
    output_path=FIGURES_ROOT / 'resnet18_gradcam_examples.png',
    device=device,
    normalize_mode='imagenet',
    max_examples=6,
)

print('Saved outputs under:', RESULTS_ROOT)
print('Baseline checkpoint:', RESULTS_ROOT / 'baseline_cnn' / 'baseline_cnn_best.pt')
print('ResNet18 checkpoint:', RESULTS_ROOT / 'resnet18' / 'resnet18_best.pt')